In [1]:
import pandas as pd
import numpy as np
import os
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import TIME_WINDOWS, TOI
from config.paths import ERPAC_DIR

Data extraction for LMMs -> **R analysis**

In [3]:
task = "DeCRAT"

erpac_df = pd.read_parquet(
    r"F:\# study 2\eeg_data\erpac\DeCRAT_erpac_results.parquet"
)

erpac_df

,sub,group,task,task_stage,task_block,coupling,roi,amp_freq,time,erpac_value
0,s1_pac_sub01,Y,DeCRAT,plan,baseline,theta_gamma,M1,32.5,-0.500,0.154193
1,s1_pac_sub01,Y,DeCRAT,plan,baseline,theta_gamma,M1,32.5,-0.498,0.151917
2,s1_pac_sub01,Y,DeCRAT,plan,baseline,theta_gamma,M1,32.5,-0.496,0.148534
3,s1_pac_sub01,Y,DeCRAT,plan,baseline,theta_gamma,M1,32.5,-0.494,0.144634
4,s1_pac_sub01,Y,DeCRAT,plan,baseline,theta_gamma,M1,32.5,-0.492,0.142908
...,...,...,...,...,...,...,...,...,...,...
55937515,s1_pac_sub68,O,DeCRAT,go,adaptation,beta_gamma,SMA,76.5,0.692,0.089748
55937516,s1_pac_sub68,O,DeCRAT,go,adaptation,beta_gamma,SMA,76.5,0.694,0.089232
55937517,s1_pac_sub68,O,DeCRAT,go,adaptation,beta_gamma,SMA,76.5,0.696,0.088614
55937518,s1_pac_sub68,O,DeCRAT,go,adaptation,beta_gamma,SMA,76.5,0.698,0.087683


AVERAGE ACROSS FREQS AND TIME WINDOWS

In [6]:
def average_erpac_windows(erpac_df, task, time_windows=TIME_WINDOWS):

    df = erpac_df.copy()

    def assign_time_window(row):

        stage = row["task_stage"]
        t = row["time"]

        for window_name, (tmin, tmax) in time_windows[task][stage].items():
            if tmin <= t < tmax:
                return window_name

        return np.nan

    df["time_win"] = df.apply(
        assign_time_window,
        axis=1
    )

    df = df.dropna(
        subset=["time_win"]
    )

    if task == "FTT":
        df_grouped = (
            df.groupby(
                [
                    "sub",
                    "group",
                    "task",
                    "task_stage",
                    "coupling",
                    "roi",
                    "time_win",
                ],
                as_index=False,
            )
            ["erpac_value"]
            .mean()
        )
    else:
        df_grouped = (
            df.groupby(
                [
                    "sub",
                    "group",
                    "task",
                    "task_stage",
                    "task_block",
                    "coupling",
                    "roi",
                    "time_win",
                ],
                as_index=False,
            )
            ["erpac_value"]
            .mean())

    return df_grouped

erpac_window_df = average_erpac_windows(erpac_df, task=task)
erpac_window_df

,sub,group,task,task_stage,task_block,coupling,roi,time_win,erpac_value
0,s1_pac_sub01,Y,DeCRAT,go,adaptation,alpha_gamma,M1,late,0.093389
1,s1_pac_sub01,Y,DeCRAT,go,adaptation,alpha_gamma,M1,middle,0.092060
2,s1_pac_sub01,Y,DeCRAT,go,adaptation,alpha_gamma,M1,start,0.099347
3,s1_pac_sub01,Y,DeCRAT,go,adaptation,alpha_gamma,PMC,late,0.093927
4,s1_pac_sub01,Y,DeCRAT,go,adaptation,alpha_gamma,PMC,middle,0.090433
...,...,...,...,...,...,...,...,...,...
6763,s1_pac_sub77,Y,DeCRAT,plan,baseline,theta_gamma,S1,late,0.191087
6764,s1_pac_sub77,Y,DeCRAT,plan,baseline,theta_gamma,S1,middle,0.202072
6765,s1_pac_sub77,Y,DeCRAT,plan,baseline,theta_gamma,SMA,early,0.203499
6766,s1_pac_sub77,Y,DeCRAT,plan,baseline,theta_gamma,SMA,late,0.203959


In [7]:
output_path = os.path.join(
    ERPAC_DIR,
    f"erpac_window_averaged_{task}.csv"
)

erpac_window_df.to_csv(
    output_path,
    index=False
)

print(erpac_window_df.head())
print(erpac_window_df.shape)


            sub group    task task_stage  task_block     coupling  roi  \
0  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma   M1   
1  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma   M1   
2  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma   M1   
3  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma  PMC   
4  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma  PMC   

  time_win  erpac_value  
0     late     0.093389  
1   middle     0.092060  
2    start     0.099347  
3     late     0.093927  
4   middle     0.090433  
(6768, 9)


AVERAGE ACROSS TIME

In [10]:
def average_erpac_stage(erpac_df, task, TOI=TOI):

    if task == "FTT":
        plan_time_range = TOI[task]["plan"]
        go_time_range = TOI[task]["go"]
        plan_start, plan_end = plan_time_range["start"], plan_time_range["end"]
        go_start, go_end = go_time_range["start"], go_time_range["end"]
    else:
        plan_time_range = TOI[task]["plan"]
        go_time_range = TOI[task]["go"]
        plan_start, plan_end = plan_time_range["start"], plan_time_range["end"]
        go_start, go_end = go_time_range["start"], go_time_range["end"]

    df = erpac_df.copy()

    # Keep only the desired time range for each stage
    mask = (
        (
            (df["task_stage"] == "plan") &
            (df["time"] >= plan_start) &
            (df["time"] <= plan_end)
        )
        |
        (
            (df["task_stage"] == "go") &
            (df["time"] >= go_start) &
            (df["time"] <= go_end)
        )
    )

    df = df[mask]

    # Average across both time and amplitude-frequency bins
    if task == "FTT":
        df_mean = (
            df
            .groupby(
                [
                    "sub",
                    "group",
                    "task",
                    "task_stage",
                    "coupling",
                    "roi",
                ],
                as_index=False,
            )["erpac_value"]
            .mean()
        )
    else:
        df_mean = (
            df
            .groupby(
                [
                "sub",
                "group",
                "task",
                "task_stage",
                "task_block",
                "coupling",
                "roi",
            ],
            as_index=False,
        )["erpac_value"]
        .mean()
    )

    return df_mean

In [11]:
task = "DeCRAT"
erpac_stage_mean_df = average_erpac_stage(erpac_df, task, TOI)

print(erpac_stage_mean_df.head())
print(erpac_stage_mean_df.shape)

output_path = os.path.join(
    ERPAC_DIR,
   f"erpac_no_time_{task}.csv"
)

erpac_stage_mean_df.to_csv(
    output_path,
    index=False
)

print(f"Saved to: {output_path}")

            sub group    task task_stage  task_block     coupling  roi  \
0  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma   M1   
1  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma  PMC   
2  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma   S1   
3  s1_pac_sub01     Y  DeCRAT         go  adaptation  alpha_gamma  SMA   
4  s1_pac_sub01     Y  DeCRAT         go  adaptation   beta_gamma   M1   

   erpac_value  
0     0.094189  
1     0.094121  
2     0.094696  
3     0.094619  
4     0.096539  
(2256, 8)
Saved to: F:\# study 2\eeg_data\erpac\erpac_no_time_DeCRAT.csv


WITH TIME WINDOWS AVERAGED ACROSS ROI

Prep done. Move to R for LMMs analysis